# **Project 2 - Sales Analysis - ETL**

## Objectives

- Extract data from provided CSV files
- Clean it 
- Apply feature engineering if necessary 
- Remove any unnecessary columns
- Save to a new CSV file


## Inputs

CSV files provided:

stores data-set.csv
sales data-set.csv
features data set.csv

Renamed files to a uniform naming convention:

Sales_Features_DataSet.csv
Sales_DataSet.csv
Sales_Stores_DataSet.csv

Note: original files are stored in Data/OriginalFiles


## Outputs

CSV file created from ETL etc stored in Data/CleanedDataSets:

Sales_Features_DataSet_Cleaned.csv



## Additional Comments

Developed an experimental ETL library which is in this project (modETL_library.py) 
Has lots of cool features so will be interesting to see how it works "in the field"

Used AI tool used to help with the ETL process:
- GitHub 
- Copilot 

See Documents/What_AI_Used_For.md for more details.


## Initalise Working Environment

In [1]:
#import libraries
import os
import numpy as np
import pandas as pd

#below solution provided by chatGPT 
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root / "assets" / "python_files") not in sys.path:
    sys.path.insert(0, str(project_root / "assets" / "python_files"))
    
import modGlobal
import modETL_Library as modETL
#end solution provided by chatGPT

# Section 1 - Initalisation

## Intitialise All Variables To Be Used In Global Stack 

In [ ]:
#DataFrame vars for ETL
dfSales_Features_DataSet = None
dfSales_Features_DataSet_Work = None
dfCleaned = None

#missing values check vars
intLessThanZero = 0

#stores current directory
strCurrentDir = ""

#other vars
dictDataFrames = dict()
lstColumns = list()

## Set Current Directory To Base Project Directory

In [3]:
#get project directory - default is jupyter notebook sub folder as that is where this file is located!
#so move back one to the project root path
# Source - https://stackoverflow.com/a/17726833
# Posted by chimpsarehungry
# Retrieved 2026-07-05, License - CC BY-SA 3.0

#get current folder
strCurrentDir = os.getcwd()

#is the last part of the path the project directory?
if not strCurrentDir.endswith(modGlobal.CNST_STR_PROJECT_DIR):
   #get current working directory and move back one to the project root path
   strCurrentDir =  os.path.normpath(os.getcwd() + os.sep + os.pardir)
   os.chdir(os.path.dirname(strCurrentDir))
   #change directory
   os.chdir(strCurrentDir)

#confirm current directory is project directory
print(f"Current Directory: \n {os.getcwd()}")

Current Directory: 
 /Users/rogerwilliams/Projects/Python/CourseProjects/Project2-SalesAnalysis


# Section 1 - Extraction

- Read csv file sales data-set.csv as other two do not require ETL (boo!)
- Move into working files directory
- Read csv file into a pandas dataframe
- Get schema info -> column and row numbers
- Get first 5 records
- Get list of column datatypes
- Get detailed schema infomation

## Read csv File Into Variable For Processing

In [4]:
#read csv file into DataFrame
dfSales_Features_DataSet = modETL.funcReadFileReturnDataFrame("Features_DataSet.csv")
#save as working csv file
modETL.funcSaveDataFrameToWorkingFile(dfSales_Features_DataSet)

#read file from working files folder
#ETL library returns a dictionary of all files in the folder with the attribute name
#set to the actual csv filename
dictDataFrames = modETL.funcReadWorkingFilesReturnDictionary()
dfSales_Features_DataSet = dictDataFrames["Features_DataSet_Working.csv"]

Folder Structure Created Successfully!
Read Features_DataSet.csv Into DataFrame

Saved: Features_DataSet To Working Folder
3 csv Files Read Into DataFrames

DataFrames Created:
Features_DataSet_Working.csv
Stores_DataSet_Working.csv
Sales_DataSet_Working.csv




## Get Schema Info - Sales_Features_DataSet

In [5]:
modETL.funcGetStructure(dfSales_Features_DataSet)
dfSales_Features_DataSet.head()

DataFrame Structure:
<class 'pandas.DataFrame'>
RangeIndex: 8190 entries, 0 to 8189
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Unnamed: 0    8190 non-null   int64  
 1   Store         8190 non-null   int64  
 2   Date          8190 non-null   str    
 3   Temperature   8190 non-null   float64
 4   Fuel_Price    8190 non-null   float64
 5   MarkDown1     4032 non-null   float64
 6   MarkDown2     2921 non-null   float64
 7   MarkDown3     3613 non-null   float64
 8   MarkDown4     3464 non-null   float64
 9   MarkDown5     4050 non-null   float64
 10  CPI           7605 non-null   float64
 11  Unemployment  7605 non-null   float64
 12  IsHoliday     8190 non-null   bool   
dtypes: bool(1), float64(9), int64(2), str(1)
memory usage: 855.9 KB
None


Summary of DataFrame Structure:
Numeric Columns:

        Unnamed: 0        Store  Temperature   Fuel_Price      MarkDown1  \
count  8190.000000  8190.000000  819

,Unnamed: 0,Store,Date,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday
0,0,1,05/02/2010,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False
1,1,1,12/02/2010,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,True
2,2,1,19/02/2010,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,False
3,3,1,26/02/2010,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,False
4,4,1,05/03/2010,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,False


## Observations - Sales_Features_DataSet
8190 rows
13 columns

Columns:

Unnamed, Store, Date, Temperature, Fuel_Price, MarkDown1, MarkDown2, MarkDown3, MarkDown4, MarkDown5, 
CPI, Unemployment, IsHoliday

Assuming Unnamed is an index column and can be dropped if necessary

We can also see the data types:

Unnamed : int64
Store : int64
Type : str
Size : int64

In the Summary of Dataframe Structure for numerical values we see this:

DataFrame Structure:
===================
<class 'pandas.DataFrame'>
RangeIndex: 8190 entries, 0 to 8189
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Unnamed: 0    8190 non-null   int64  
 1   Store         8190 non-null   int64  
 2   Date          8190 non-null   str    
 3   Temperature   8190 non-null   float64
 4   Fuel_Price    8190 non-null   float64
 5   MarkDown1     4032 non-null   float64
 6   MarkDown2     2921 non-null   float64
 7   MarkDown3     3613 non-null   float64
 8   MarkDown4     3464 non-null   float64
 9   MarkDown5     4050 non-null   float64
 10  CPI           7605 non-null   float64
 11  Unemployment  7605 non-null   float64
 12  IsHoliday     8190 non-null   bool   
dtypes: bool(1), float64(9), int64(2), str(1)
memory usage: 775.9 KB
None

Summary of DataFrame Structure:
===============================
Numeric Columns:

        Unnamed: 0        Store  Temperature   Fuel_Price      MarkDown1  \
count  8190.000000  8190.000000  8190.000000  8190.000000    4032.000000   
mean   4094.500000    23.000000    59.356198     3.405992    7032.371786   
std    2364.393685    12.987966    18.678607     0.431337    9262.747448   
min       0.000000     1.000000    -7.290000     2.472000   -2781.450000   
25%    2047.250000    12.000000    45.902500     3.041000    1577.532500   
50%    4094.500000    23.000000    60.710000     3.513000    4743.580000   
75%    6141.750000    34.000000    73.880000     3.743000    8923.310000   
max    8189.000000    45.000000   101.950000     4.468000  103184.980000   

           MarkDown2      MarkDown3     MarkDown4      MarkDown5          CPI  \
count    2921.000000    3613.000000   3464.000000    4050.000000  7605.000000   
mean     3384.176594    1760.100180   3292.935886    4132.216422   172.460809   
std      8793.583016   11276.462208   6792.329861   13086.690278    39.738346   
min      -265.760000    -179.260000      0.220000    -185.170000   126.064000   
25%        68.880000       6.600000    304.687500    1440.827500   132.364839   
50%       364.570000      36.260000   1176.425000    2727.135000   182.764003   
75%      2153.350000     163.150000   3310.007500    4832.555000   213.932412   
max    104519.540000  149483.310000  67474.850000  771448.100000   228.976456   

       Unemployment  
count   7605.000000  
mean       7.826821  
std        1.877259  
min        3.684000  
25%        6.634000  
50%        7.806000  
75%        8.567000  
max       14.313000  

**What Does This Mean?**

- Count is how many values are in the columns
- Mean is the average value of the column
- Std is the standard deviation of the column 
  Standard deviation is how far from the average (mean) the values are. The closer to 0 the
  more consistent the values are i.e. not much deviation from the mean. The deviation range 
  can be referred to as the Sigma. The measurement is based on the Alpha this is a percentage
  of the mean used to determine variance. ypically this is set to 5%
- Min is the minimum value of the column
- 25% is the value at the 25th percentile of the column
- 50% is the value between the 25th and 75th percentiles of the column
- 75% is the value at the 75th percentile of the column

The middle 50% is also called the Interquartile Range (IQR) and its width can be set by changing the
alpha value which is normally 5%

**What?**

Percentile is a technical term for a 25% segment of the data. 
50% is the mean, 25% is below that and 75% is 25%above that, this crude picture illustrates this:

 1st Percentile      2nd Percentile        3rd Percentile
0      -      25%  25%      -     75%   75%      -     100%  

Note: 2nd percentile can also be called Central Tendency

- Max is the maximum value of the column

String Columns:

              Date
count         8190 - number of records  
unique         182 - number of unique values
top     05/02/2010 - most frequent value 
freq            45 - frequency of most frequent value


# Get Column Data Types

In [6]:
#get column data types
print (f"Data Types: \n{dfSales_Features_DataSet.dtypes}")

Data Types: 
Unnamed: 0        int64
Store             int64
Date                str
Temperature     float64
Fuel_Price      float64
MarkDown1       float64
MarkDown2       float64
MarkDown3       float64
MarkDown4       float64
MarkDown5       float64
CPI             float64
Unemployment    float64
IsHoliday          bool
dtype: object


## Observations - Sales_Features_DataSet

- Date is a string - needs conversion to datetime. 
- Other column types look ok

## Get Schema Statistics

Look for duplicates and missing values

In [7]:
#get column statistics
modETL.funcGetStatistics(dfSales_Features_DataSet)

DataFrame Statistics:
        Unnamed: 0        Store  Temperature   Fuel_Price      MarkDown1  \
count  8190.000000  8190.000000  8190.000000  8190.000000    4032.000000   
mean   4094.500000    23.000000    59.356198     3.405992    7032.371786   
std    2364.393685    12.987966    18.678607     0.431337    9262.747448   
min       0.000000     1.000000    -7.290000     2.472000   -2781.450000   
25%    2047.250000    12.000000    45.902500     3.041000    1577.532500   
50%    4094.500000    23.000000    60.710000     3.513000    4743.580000   
75%    6141.750000    34.000000    73.880000     3.743000    8923.310000   
max    8189.000000    45.000000   101.950000     4.468000  103184.980000   

           MarkDown2      MarkDown3     MarkDown4      MarkDown5          CPI  \
count    2921.000000    3613.000000   3464.000000    4050.000000  7605.000000   
mean     3384.176594    1760.100180   3292.935886    4132.216422   172.460809   
std      8793.583016   11276.462208   6792.329861 

## Observations - Sales_Features_DataSet

Date is a string - needs conversion to datetime  
Other column types look ok
If unnamed causes issues later on will drop the column!

Observations:
- High levels of missing values in markdown columns which will need addressing
- High duplicates but this is to be expected as store number/date and department would be repeated


---

## Observations - Sales_Features_DataSet

Missing values for columns:
MarkDown1, MarkDown2, MarkDown3, MarkDown4, MarkDown5
CPI, Employment

CPI and Employment are not used in this analysis so will be dropped from the dataset later on

## Get Unique Values - Sales_Features_DataSet

In [8]:
#show unique values count
modETL.funcGetUniqueValuesCount(dfSales_Features_DataSet)

DataFrame Unique Values Per Column:
Unnamed: 0           - 8190: Unique Values Out Of 8190 Total Values
Store                - 45: Unique Values Out Of 8190 Total Values
Date                 - 182: Unique Values Out Of 8190 Total Values
Temperature          - 4178: Unique Values Out Of 8190 Total Values
Fuel_Price           - 1011: Unique Values Out Of 8190 Total Values
MarkDown1            - 4023: Unique Values Out Of 4032 Total Values
MarkDown2            - 2715: Unique Values Out Of 2921 Total Values
MarkDown3            - 2885: Unique Values Out Of 3613 Total Values
MarkDown4            - 3405: Unique Values Out Of 3464 Total Values
MarkDown5            - 4045: Unique Values Out Of 4050 Total Values
CPI                  - 2505: Unique Values Out Of 7605 Total Values
Unemployment         - 404: Unique Values Out Of 7605 Total Values
IsHoliday            - 2: Unique Values Out Of 8190 Total Values



## Observations - Sales_Features_DataSet

Store has 45 unique values  
Date has 182 unique values  
Temperature has 4178 unique values
Fuel_Price has 1011 unique values
MarkDown1 has 4023 unique values
MarkDown2 has 2715 unique values
MarkDown3 has 2885 unique values
MarkDown4 has 3405 unique values
MarkDown5 has 4045 unique values
CPI has 2505 unique values
Unemployment has 404 unique values
IsHoliday has 2 unique values



## Get Categorical Value Distribution

In [9]:
#get categorical distribution  no needed as only one string value: Date
#and that will converted to datetime later on


# Section 2

- If missing values determine what to fill with (mean, mode or categorical something else)
- Transform data types if necessary


In [10]:
## Create Copy Of DataFrame
dfSales_Features_DataSet_Work = dfSales_Features_DataSet.copy()

## Replace Missing Values With ZERO For MarkDown Columns

In [11]:
#replace missing values for markdown columns with zero. 
#Copilot created code applied naming conventions

lstMarkDownColumns = ["MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5"]
#replace missing values
dfSales_Features_DataSet_Work[lstMarkDownColumns] = dfSales_Features_DataSet_Work[lstMarkDownColumns].fillna(0)

# verify
dfSales_Features_DataSet_Work[lstMarkDownColumns].isna().sum()

MarkDown1    0
MarkDown2    0
MarkDown3    0
MarkDown4    0
MarkDown5    0
dtype: int64

## Observations - Sales_Features_DataSet

Worked! Thanks Copilot, no missing values in the MarkDown numeric columns

Missing values remain the same yet the columns are now without NaN/null values
At this point not concerned, at least the data is clean...

## Strip Spaces From Date Column

In [12]:
#remove spaces from Date column (string datatype)
for objColumn in ["Date"]:
    dfSales_Features_DataSet_Work[objColumn] = dfSales_Features_DataSet_Work[objColumn].str.strip()
    
#show first 50 rows to check    
dfSales_Features_DataSet_Work.head(50)    

,Unnamed: 0,Store,Date,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday
0,0,1,05/02/2010,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.096358,8.106,False
1,1,1,12/02/2010,38.51,2.548,0.0,0.0,0.0,0.0,0.0,211.242170,8.106,True
2,2,1,19/02/2010,39.93,2.514,0.0,0.0,0.0,0.0,0.0,211.289143,8.106,False
3,3,1,26/02/2010,46.63,2.561,0.0,0.0,0.0,0.0,0.0,211.319643,8.106,False
4,4,1,05/03/2010,46.50,2.625,0.0,0.0,0.0,0.0,0.0,211.350143,8.106,False
5,5,1,12/03/2010,57.79,2.667,0.0,0.0,0.0,0.0,0.0,211.380643,8.106,False
6,6,1,19/03/2010,54.58,2.720,0.0,0.0,0.0,0.0,0.0,211.215635,8.106,False
7,7,1,26/03/2010,51.45,2.732,0.0,0.0,0.0,0.0,0.0,211.018042,8.106,False
8,8,1,02/04/2010,62.27,2.719,0.0,0.0,0.0,0.0,0.0,210.820450,7.808,False
9,9,1,09/04/2010,65.86,2.770,0.0,0.0,0.0,0.0,0.0,210.622857,7.808,False


## Observations - Sales_Features_DataSet

No data corruptions

## Validate Numerical Values For Columns Used In Analysis

- see if Stores column has values less than 1
- see if MarkDown1 column has values less than 1
- see if MarkDown2 column has values less than 1
- see if MarkDown3 column has values less than 1
- see if MarkDown4 column has values less than 1
- see if MarkDown5 column has values less than 1


In [13]:
#get count of how many values in Stores column are less than 1
intLessThanOne = (dfSales_Features_DataSet_Work["Store"] < 1).sum()
#print result
intLessThanOne

np.int64(0)

---

## Observations - Sales_Features_DataSet

Values for Store in acceptable limits

## Check All MarkDown Columns For Values Less Than 0

In [14]:
#check MarkDown columns for < 0

for intNum in range(1,6):
    intLessThanOne = (dfSales_Features_DataSet_Work[ f"MarkDown{intNum}" ] < 0).sum()
    #print result
    print( f"MarkDown Column {intNum} Has Values Less Than Zero? {intLessThanOne > 0}")
    
    
#Check findings    
dfSales_Features_DataSet_Work.query( "MarkDown1 <0 or MarkDown2 <0 or MarkDown3 <0 or MarkDown4 <0 or MarkDown5 <0")    

MarkDown Column 1 Has Values Less Than Zero? True
MarkDown Column 2 Has Values Less Than Zero? True
MarkDown Column 3 Has Values Less Than Zero? True
MarkDown Column 4 Has Values Less Than Zero? False
MarkDown Column 5 Has Values Less Than Zero? True


,Unnamed: 0,Store,Date,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday
657,657,4,23/03/2012,59.07,3.759,8806.80,-10.50,5.99,739.14,4396.97,130.896645,4.607,False
860,860,5,17/08/2012,87.52,3.571,1649.56,-10.98,2.31,1955.75,1205.23,222.627675,5.603,False
873,873,5,16/11/2012,56.89,3.252,1631.01,-35.74,15.46,326.59,2310.83,224.106624,5.422,False
893,893,5,05/04/2013,61.88,3.583,9023.29,927.32,170.24,405.30,-185.17,225.682320,5.278,False
1063,1063,6,11/01/2013,48.26,3.243,5510.29,33063.57,-0.86,693.10,3418.77,225.832011,5.372,False
1265,1265,7,31/05/2013,50.70,3.870,2350.78,-7.76,91.59,166.86,540.89,NaN,NaN,False
1587,1587,9,10/08/2012,88.66,3.494,3180.78,-9.94,1.40,2112.03,2314.58,225.717009,5.277,False
1591,1591,9,07/09/2012,87.93,3.730,4837.99,-5.96,22.74,602.80,1377.59,226.210354,5.277,True
1611,1611,9,25/01/2013,49.14,3.227,200.46,348.20,-14.29,59.00,2162.77,228.030618,5.049,False
1615,1615,9,22/02/2013,45.91,3.597,2810.86,1405.04,-179.26,63.12,3593.56,228.205445,5.049,False


## Observations - Sales_Features_DataSet

MarkDown columns 1, 2, 3 and 5 have values less than 0
Being as markdowns can be negative by nature am happy to ignore

## Data Transformations/Feature Engineering

Convert Date column to datetime format

In [15]:
#convert Date column to datetime format
dfSales_Features_DataSet_Work["Date"] = pd.to_datetime(dfSales_Features_DataSet["Date"], format="%d/%m/%Y")

#check results
dfSales_Features_DataSet_Work.head(50)

,Unnamed: 0,Store,Date,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday
0,0,1,2010-02-05,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.096358,8.106,False
1,1,1,2010-02-12,38.51,2.548,0.0,0.0,0.0,0.0,0.0,211.242170,8.106,True
2,2,1,2010-02-19,39.93,2.514,0.0,0.0,0.0,0.0,0.0,211.289143,8.106,False
3,3,1,2010-02-26,46.63,2.561,0.0,0.0,0.0,0.0,0.0,211.319643,8.106,False
4,4,1,2010-03-05,46.50,2.625,0.0,0.0,0.0,0.0,0.0,211.350143,8.106,False
5,5,1,2010-03-12,57.79,2.667,0.0,0.0,0.0,0.0,0.0,211.380643,8.106,False
6,6,1,2010-03-19,54.58,2.720,0.0,0.0,0.0,0.0,0.0,211.215635,8.106,False
7,7,1,2010-03-26,51.45,2.732,0.0,0.0,0.0,0.0,0.0,211.018042,8.106,False
8,8,1,2010-04-02,62.27,2.719,0.0,0.0,0.0,0.0,0.0,210.820450,7.808,False
9,9,1,2010-04-09,65.86,2.770,0.0,0.0,0.0,0.0,0.0,210.622857,7.808,False


## Remove Unnecessary Columns

In [16]:
#going to remove:
#fuel_price, CPI
dfSales_Features_DataSet_Work.drop(columns=["Fuel_Price", "CPI"], inplace=True)

#check results
dfSales_Features_DataSet_Work.info()

<class 'pandas.DataFrame'>
RangeIndex: 8190 entries, 0 to 8189
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Unnamed: 0    8190 non-null   int64         
 1   Store         8190 non-null   int64         
 2   Date          8190 non-null   datetime64[us]
 3   Temperature   8190 non-null   float64       
 4   MarkDown1     8190 non-null   float64       
 5   MarkDown2     8190 non-null   float64       
 6   MarkDown3     8190 non-null   float64       
 7   MarkDown4     8190 non-null   float64       
 8   MarkDown5     8190 non-null   float64       
 9   Unemployment  7605 non-null   float64       
 10  IsHoliday     8190 non-null   bool          
dtypes: bool(1), datetime64[us](1), float64(7), int64(2)
memory usage: 648.0 KB


## Observations - Sales_Features_DataSet

Columns confirmed removed:

Fuel_Price, CPI, Unemployment

## Save Work DataFrame To CSV File

In [17]:
#create list of columns new and old JUST what is actually needed
lstColumns = [
                #original columns
                "Store","Date","Temperature",
                "MarkDown1","MarkDown2","MarkDown3","MarkDown4","MarkDown5",
                "IsHoliday", "Unemployment"
              ]

#create new DataFrame for cleaned data
dfCleaned = dfSales_Features_DataSet_Work[lstColumns].copy()

modETL.funcSaveDataFrameToCleanedFile(dfCleaned)

#Note: excluded column: Dept as not in the hypothesis analysis which is a nice way
#      avoiding the fact I don't know just how to plot a chart with 98 departments
#      and have it readable!

Saved: Features_DataSet To Visualisation Folder


# Conclusions and Next Steps

ETL went well, DataSet ready for use